# fAIr Utilities - Package Test

Runs the full pipeline (preprocess → train → predict → polygonize) for both **RAMP** and **YOLO** models.

In [ ]:
import os
import time
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

workspace = os.getcwd()
base_path = f"{workspace}/ramp-data/sample_2"

## Preprocess

In [ ]:
from hot_fair_utilities import polygonize, predict, preprocess

preprocess_output = f"{base_path}/preprocessed"

start = time.perf_counter()
preprocess(
    input_path=f"{base_path}/input",
    output_path=preprocess_output,
    rasterize=True,
    rasterize_options=["binary"],
    georeference_images=True,
    multimasks=True,
)
print(f"Preprocessing took {time.perf_counter() - start:.2f}s")

## RAMP

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["RAMP_HOME"] = workspace

import tensorflow as tf
from hot_fair_utilities import patch_tf_experimental_layers
from hot_fair_utilities.training.ramp import train as train_ramp

patch_tf_experimental_layers()

print(f"TensorFlow {tf.__version__} | GPUs: {len(tf.config.experimental.list_physical_devices('GPU'))}")

In [ ]:
start = time.perf_counter()

accuracy, model_path = train_ramp(
    input_path=preprocess_output,
    output_path=f"{base_path}/train",
    epoch_size=1,
    batch_size=2,
    model="ramp",
    model_home=os.environ["RAMP_HOME"],
)
print(f"Accuracy: {accuracy}")
print(f"Model: {model_path}")
print(f"Training took {time.perf_counter() - start:.2f}s")

In [ ]:
prediction_output = f"{base_path}/prediction/output"

predict(
    checkpoint_path=model_path,
    input_path=f"{base_path}/prediction/input",
    prediction_path=prediction_output,
)
polygonize(
    input_path=prediction_output,
    output_path=f"{prediction_output}/prediction.geojson",
    remove_inputs=True,
)
print("RAMP predict + polygonize done")

## YOLO

In [ ]:
from hot_fair_utilities.preprocessing.yolo_v8.yolo_format import yolo_format
from hot_fair_utilities.training.yolo_v8.train import train as train_yolo

# Re-preprocess without multimasks for YOLO
preprocess(
    input_path=f"{base_path}/input",
    output_path=preprocess_output,
    rasterize=True,
    rasterize_options=["binary"],
    georeference_images=True,
    multimasks=False,
    epsg=4326,
)

yolo_data_dir = f"{base_path}/yolo_v2"

start = time.perf_counter()
yolo_format(
    input_path=preprocess_output,
    output_path=yolo_data_dir,
)
print(f"YOLO format conversion took {time.perf_counter() - start:.2f}s")

In [ ]:
start = time.perf_counter()

model_path, iou_accuracy = train_yolo(
    data=base_path,
    weights=f"{workspace}/yolov8s_v2-seg.pt",
    epochs=2,
    batch_size=16,
    pc=2.0,
    output_path=yolo_data_dir,
    dataset_yaml_path=os.path.join(yolo_data_dir, "yolo_dataset.yaml"),
)
print(f"IoU Accuracy: {iou_accuracy}")
print(f"Training took {time.perf_counter() - start:.2f}s")

In [ ]:
prediction_output = f"{base_path}/prediction/output"

start = time.perf_counter()
predict(
    checkpoint_path=model_path,
    input_path=f"{base_path}/prediction/input",
    prediction_path=prediction_output,
)
polygonize(
    input_path=prediction_output,
    output_path=f"{prediction_output}/prediction.geojson",
    remove_inputs=False,
)
print(f"YOLO predict + polygonize took {time.perf_counter() - start:.2f}s")